# Null test — does P-gate(0.05)'s positive extension Sharpe survive against random day-selection?

**Question.** ENS1 P-gate(0.05) earns a post-cost Sharpe of approximately +0.94 on the 792 days it chooses to trade during the 2015-2025 extension. The other ~68 % of days it sits out (no-trade-band fill). Is that +0.94 driven by the model identifying days on which it has real signal, or by a sampling artifact — i.e., would a random 792-day subsample of the underlying P-only series have produced a similar Sharpe?

**Method.** Build a null distribution by repeatedly sampling 792 distinct days (without replacement) from the ENS1 P-only daily return series in the same era, computing the annualized Sharpe on each subsample. Compare the observed gate Sharpe to this distribution.

**Conventions.** Sharpe = √252 · mean / std with `ddof=1`. Same formula applied to the observed gate series and every null draw. No risk-free subtraction (raw Sharpe), applied symmetrically. Seed: `np.random.default_rng(42)`. Trials: 10,000.

In [1]:
# Setup: imports and load the equity-curves parquet.
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path('..')
equity = pd.read_parquet(ROOT / 'app' / 'data' / 'equity_curves.parquet')

# Filter to the (era, model, cost) cell we care about. Both P-gate(0.05) and
# P-only live in this same sub-table.
base = equity[
    (equity['era'] == '2015-2025 (extension)')
    & (equity['cost_regime'] == '5bps_half_turn')
    & (equity['model'] == 'ENS1')
].copy()

print('Sub-table per scheme:')
print(base.groupby('scheme').agg(
    n_days=('ret', 'size'),
    n_nonzero=('ret', lambda x: (x != 0).sum()),
    mean=('ret', 'mean'),
    std=('ret', 'std'),
).round(6))

Sub-table per scheme:
              n_days  n_nonzero      mean       std
scheme                                             
P-gate(0.03)    2500       2266  0.000651  0.027605
P-gate(0.05)    2500        792  0.001022  0.030847
P-only          2500       2500 -0.000428  0.017297
Product         2500       2500 -0.001382  0.015613
U-only          2500       2500 -0.000728  0.017670
Z-comp          2500       2500 -0.000790  0.018205


In [2]:
# Null test: observed gate Sharpe + 10,000 random-subsample null distribution.
gate = base[base['scheme'] == 'P-gate(0.05)'].sort_values('date').reset_index(drop=True)
pony = base[base['scheme'] == 'P-only'].sort_values('date').reset_index(drop=True)

gate_returns = gate['ret'].values
pony_returns = pony['ret'].values

# P-gate(0.05) is densified to the full 2,500-day trading calendar with zero
# fill on no-trade days. Sharpe must be computed on the actually-traded days.
nonzero_mask = gate_returns != 0.0
gate_nonzero = gate_returns[nonzero_mask]

ann_factor = np.sqrt(252)
gate_sharpe = ann_factor * gate_nonzero.mean() / gate_nonzero.std(ddof=1)

# Null distribution: sample 792 distinct days without replacement from P-only
# and compute Sharpe with the identical formula.
N_SAMPLE = len(gate_nonzero)         # 792
N_TRIALS = 10_000
total_days = len(pony_returns)       # 2,500
rng = np.random.default_rng(seed=42)

null_sharpes = np.empty(N_TRIALS)
for i in range(N_TRIALS):
    idx = rng.choice(total_days, size=N_SAMPLE, replace=False)
    sub = pony_returns[idx]
    null_sharpes[i] = ann_factor * sub.mean() / sub.std(ddof=1)

In [3]:
# Report: gate_sharpe, null distribution stats, percentile, interpretation.
print(f'P-gate(0.05) total days: {len(gate_returns)}')
print(f'P-gate(0.05) non-zero days: {len(gate_nonzero)}')
print(f'P-only total days: {len(pony_returns)}')

print(f'\ngate_sharpe (sqrt(252) * mean/std on {len(gate_nonzero)} non-zero days) = {gate_sharpe:.4f}')
print(f'  mean daily ret: {gate_nonzero.mean():.6f}')
print(f'  std  daily ret: {gate_nonzero.std(ddof=1):.6f}')

print(f'\nNull distribution ({N_TRIALS:,} trials, sample size = {N_SAMPLE}, seed=42):')
print(f'  mean   : {null_sharpes.mean():.4f}')
print(f'  std    : {null_sharpes.std(ddof=1):.4f}')
print(f'  P05    : {np.percentile(null_sharpes, 5):.4f}')
print(f'  P50    : {np.percentile(null_sharpes, 50):.4f}')
print(f'  P95    : {np.percentile(null_sharpes, 95):.4f}')
print(f'  P99    : {np.percentile(null_sharpes, 99):.4f}')

pct = (null_sharpes < gate_sharpe).mean() * 100
print(f'\ngate_sharpe ({gate_sharpe:.4f}) is at the {pct:.2f} percentile of the null distribution.')
print(f'Only {int((null_sharpes >= gate_sharpe).sum())} of {N_TRIALS:,} random subsamples produced a higher Sharpe.')

if pct >= 95:
    interp = 'EVIDENCE OF SIGNAL: gate_sharpe lies at the {0:.1f}th percentile, above the 95th-pct threshold — the P-gate(0.05) Sharpe is unlikely to be a random day-selection artifact.'.format(pct)
elif pct >= 75:
    interp = 'WEAK EVIDENCE: gate_sharpe sits at the {0:.1f}th percentile — somewhat elevated, but not conclusively above what random day-picking would produce.'.format(pct)
elif pct >= 25:
    interp = 'NO EVIDENCE OF SIGNAL: gate_sharpe is near the median ({0:.1f}th percentile) — indistinguishable from random day-selection from the P-only return distribution.'.format(pct)
else:
    interp = 'WORSE THAN RANDOM: gate_sharpe at the {0:.1f}th percentile is below what random day-picking typically yields.'.format(pct)

print(f'\nInterpretation: {interp}')

P-gate(0.05) total days: 2500
P-gate(0.05) non-zero days: 792
P-only total days: 2500

gate_sharpe (sqrt(252) * mean/std on 792 non-zero days) = 0.9348
  mean daily ret: 0.003225
  std  daily ret: 0.054763

Null distribution (10,000 trials, sample size = 792, seed=42):
  mean   : -0.4055
  std    : 0.4822
  P05    : -1.2192
  P50    : -0.3951
  P95    : 0.3735
  P99    : 0.6562

gate_sharpe (0.9348) is at the 99.83 percentile of the null distribution.
Only 17 of 10,000 random subsamples produced a higher Sharpe.

Interpretation: EVIDENCE OF SIGNAL: gate_sharpe lies at the 99.8th percentile, above the 95th-pct threshold — the P-gate(0.05) Sharpe is unlikely to be a random day-selection artifact.


## Summary

Observed gate Sharpe **0.9348** sits at the **99.83rd percentile** of the null distribution built from 10,000 random 792-day subsamples of ENS1 P-only post-cost returns in the same era. Only 17 of 10,000 random subsamples produced a higher Sharpe. The null distribution itself is centered at −0.40 (mirroring P-only's negative post-cost mean in the extension era), with a 99th percentile of 0.66 — below the observed gate Sharpe.

**Reading.** The gate is doing real work. The +0.94 cannot be plausibly attributed to having drawn a lucky 792-day subsample from a noisy distribution: it lies ~2.8 std above the null mean.

**Caveat.** This null tests *day-selection*, not stock-selection. It rules out the hypothesis that the observed Sharpe is a sampling fluke from the underlying P-only return distribution. It does not distinguish between the two favorable readings:

1. The model still has a rare strong edge on a minority of days, and the gate isolates them.
2. The model has a small persistent edge across all days, and the gate amplifies it by trading less often (lower turnover → less cost drag → higher post-cost Sharpe on the days it does trade).

Both interpretations are consistent with the +0.94 result; the null simply rules out the unfavorable third reading ("random luck on a small subsample").